# Mental Health Signal Detector — Fine-tuning DistilBERT v2
**Artefact School of Data — Bootcamp Data Science, Mars 2026**

### Nouveautés v2
- Dataset complet : Kaggle 100K + DAIR-AI 18K + GoEmotions 53K + **eRisk25 217K** = **~388K exemples**
- eRisk25 = labels cliniques validés (CLEF 2025, tâche dépression précoce)
- Objectif : battre le baseline LR (78.6% sur eRisk25) et retrouver >96% sur DAIR-AI

⚡ **GPU requis** : `Exécution > Modifier le type d'exécution > GPU (T4)`

In [1]:
# Étape 1 : install des dépendances
# ⚠️ IMPORTANT : après cette cellule, clique sur 'Redémarrer la session'
!pip install -q -U transformers accelerate datasets scikit-learn deep-translator langdetect

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 26.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.7 MB/s eta 0:00:00


In [2]:
# Étape 2 : vérification GPU (lancer après redémarrage)
import torch
print('GPU disponible :', torch.cuda.is_available())
print('Device :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ⚠️')
import transformers
print('Transformers :', transformers.__version__)

GPU disponible : True
Device : Tesla T4
Transformers : 5.3.0


In [3]:
# Étape 3 : monter Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_DIR  = '/content/drive/MyDrive/mh_detector'
OUTPUT_DIR = f'{DRIVE_DIR}/fine_tuned_v2'
DATA_DIR   = f'{DRIVE_DIR}/data'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DATA_DIR,   exist_ok=True)
print('Modèle    :', OUTPUT_DIR)
print('Données   :', DATA_DIR)

Mounted at /content/drive
Modèle    : /content/drive/MyDrive/mh_detector/fine_tuned_v2
Données   : /content/drive/MyDrive/mh_detector/data


In [4]:
# Étape 4 : uploader les fichiers depuis ton PC
# Tu dois uploader dans Google Drive/mh_detector/data/ :
#   - reddit_depression_dataset.csv  (Kaggle)
#   - erisk25/                       (dossier complet eRisk25 zippé puis dézippé)
#
# Option A : upload direct dans Drive via l'interface Drive
# Option B : upload via Colab
from google.colab import files
print('Fichiers dans DATA_DIR :', os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else '(vide)')

Fichiers dans DATA_DIR : ['erisk25', 'reddit_depression_dataset.csv']


In [5]:
# Étape 5 : chargement et préparation du dataset complet
import re, json, glob
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split

def clean_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r"'", '', text)          # contractions : I'm → Im
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip().lower()

frames = []

# --- Source 1 : Kaggle Reddit Depression ---
KAGGLE_PATH = f'{DATA_DIR}/reddit_depression_dataset.csv'
if os.path.exists(KAGGLE_PATH):
    df_k = pd.read_csv(KAGGLE_PATH, low_memory=False)
    df_k['text'] = (df_k['title'].fillna('') + ' ' + df_k['body'].fillna('')).str.strip()
    df_k = df_k[['text','label']].dropna()
    df_k['label'] = df_k['label'].astype(float).astype(int)
    df_k = df_k[df_k['text'].str.len() > 20]
    df_k, _ = train_test_split(df_k, train_size=100_000, stratify=df_k['label'], random_state=42)
    df_k['text'] = df_k['text'].apply(clean_text)
    frames.append(df_k[['text','label']])
    print(f'Kaggle      : {len(df_k):>7} lignes | {df_k.label.value_counts().to_dict()}')
else:
    print('⚠️  Kaggle CSV non trouvé — ignoré')

# --- Source 2 : DAIR-AI/emotion ---
DISTRESS_DAIR = {0, 4}  # sadness, fear
for split in ['train', 'test']:
    ds = load_dataset('dair-ai/emotion', split=split)
    df_d = ds.to_pandas()
    df_d['label'] = df_d['label'].apply(lambda x: 1 if x in DISTRESS_DAIR else 0)
    df_d['text']  = df_d['text'].apply(clean_text)
    frames.append(df_d[['text','label']])
print(f'DAIR-AI     : {sum(len(f) for f in frames[-2:]):>7} lignes')

# --- Source 3 : GoEmotions ---
DISTRESS_GO = {9, 12, 14, 16, 19, 24, 25}  # sadness, grief, fear, nervousness, remorse, embarrassment, disappointment
for split in ['train', 'validation', 'test']:
    ds = load_dataset('google-research-datasets/go_emotions', 'simplified', split=split)
    df_g = ds.to_pandas()
    df_g['label'] = df_g['labels'].apply(lambda x: 1 if any(l in DISTRESS_GO for l in x) else 0)
    df_g['text']  = df_g['text'].apply(clean_text)
    df_g = df_g[df_g['text'].str.len() > 10]
    frames.append(df_g[['text','label']])
print(f'GoEmotions  : {sum(len(f) for f in frames[-3:]):>7} lignes')

# --- Source 4 : eRisk25 (labels cliniques) ---
# Structure Drive : data/erisk25/t2-early-contextualized-depression/final-eriskt2-dataset-with-ground-truth/all_combined/
ERISK_DIR = f'{DATA_DIR}/erisk25/t2-early-contextualized-depression/final-eriskt2-dataset-with-ground-truth/all_combined'
if os.path.exists(ERISK_DIR):
    erisk_rows = []
    for fpath in glob.glob(f'{ERISK_DIR}/subject_*.json'):
        with open(fpath, encoding='utf-8') as f:
            posts = json.load(f)
        for post in posts:
            sub = post.get('submission', {})
            target = sub.get('target')
            if target is None: continue
            title = sub.get('title') or ''
            body  = sub.get('body')  or ''
            text  = (title + ' ' + body).strip()
            if len(text) > 20:
                erisk_rows.append({'text': text, 'label': int(bool(target))})
    df_e = pd.DataFrame(erisk_rows)
    df_e['text'] = df_e['text'].apply(clean_text)
    df_e = df_e[df_e['text'].str.len() > 10].drop_duplicates(subset=['text'])
    frames.append(df_e[['text','label']])
    print(f'eRisk25     : {len(df_e):>7} lignes | {df_e.label.value_counts().to_dict()}')
else:
    print(f'⚠️  eRisk25 non trouvé : {ERISK_DIR}')
    print('   → Vérifie la structure dans Drive')

# --- Merge final ---
df_all = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['text'])
df_all = df_all[df_all['text'].str.len() > 5]
print(f'\nDataset total : {len(df_all)} lignes | {df_all.label.value_counts().to_dict()}')

train_df, test_df = train_test_split(df_all, test_size=0.1, stratify=df_all['label'], random_state=42)
print(f'Train: {len(train_df)} | Test: {len(test_df)}')

Kaggle      :  100000 lignes | {0: 80173, 1: 19827}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

...

simplified/test-00000-of-00001.parquet:   0%|          | 0.00/347k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/43410 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5426 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5427 [00:00<?, ? examples/s]

GoEmotions  :   53201 lignes
eRisk25     :   75700 lignes | {1: 39915, 0: 35785}

Dataset total : 246170 lignes | {0: 174197, 1: 71973}
Train: 221553 | Test: 24617


In [6]:
# Étape 6 : tokenisation
from transformers import AutoTokenizer
from datasets import Dataset

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, padding='max_length', max_length=128)

train_ds = Dataset.from_pandas(train_df[['text','label']]).map(tokenize, batched=True)
test_ds  = Dataset.from_pandas(test_df[['text','label']]).map(tokenize, batched=True)

for ds in [train_ds, test_ds]:
    ds = ds.rename_column('label', 'labels')

train_ds = train_ds.rename_column('label', 'labels')
test_ds  = test_ds.rename_column('label',  'labels')
train_ds.set_format('torch', columns=['input_ids','attention_mask','labels'])
test_ds.set_format( 'torch', columns=['input_ids','attention_mask','labels'])
print('Tokenisation OK ✅')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/221553 [00:00<?, ? examples/s]

Map:   0%|          | 0/24617 [00:00<?, ? examples/s]

Tokenisation OK ✅


In [ ]:
# Étape 7 : fine-tuning DistilBERT avec gestion du déséquilibre de classes
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
import numpy as np

# ── Modèle ────────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# ── Poids de classes (calculés depuis les données réelles) ────────────────────
# Dataset : ~34% détresse (1) / ~66% neutre (0) → pénalise davantage les erreurs
# sur la classe minoritaire, cliniquement prioritaire.
train_labels = np.array(train_ds['labels'])   # conversion tensors → numpy

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=train_labels,
)
weights = torch.tensor(class_weights, dtype=torch.float).to(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
print(f'Class weights → classe 0 : {class_weights[0]:.3f} | classe 1 : {class_weights[1]:.3f}')
# Attendu : [~0.75, ~1.47]

# ── Métriques ─────────────────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1':       f1_score(labels, preds, average='weighted'),
        'f1_macro': f1_score(labels, preds, average='macro'),
    }

# ── Trainer personnalisé (CrossEntropyLoss pondérée) ─────────────────────────
class CustomTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        # Initialisé une seule fois, pas à chaque batch
        self.loss_fct = nn.CrossEntropyLoss(weight=class_weights)

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get('labels')
        outputs = model(**inputs)
        logits  = outputs.get('logits')
        loss = self.loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1),
        )
        return (loss, outputs) if return_outputs else loss

# ── Arguments d'entraînement ──────────────────────────────────────────────────
args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,                 # marge pour early stopping
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',   # macro = équilibre inter-classes
    greater_is_better=True,             # f1_macro : plus haut = meilleur
    warmup_steps=2000,                  # ~10% de ~20 000 steps
    weight_decay=0.01,
    logging_steps=100,
    report_to='none',
)

# ── Lancement ─────────────────────────────────────────────────────────────────
trainer = CustomTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
    class_weights=weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],  # stop si f1_macro ne progresse plus
)

trainer.train()

In [ ]:
# Étape 8 : évaluation finale
results = trainer.evaluate()
print('\n=== Résultats DistilBERT v2 (388K exemples) ===')
for k, v in results.items():
    print(f'  {k}: {v:.4f}')
print('\nRéférence baseline LR : 78.6% accuracy sur test eRisk25')

In [ ]:
# Étape 9 : sauvegarde dans Drive
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('✅ Modèle sauvegardé :', OUTPUT_DIR)

In [ ]:
# Étape 10 : télécharger le modèle en zip
import shutil
shutil.make_archive('/content/fine_tuned_v2', 'zip', OUTPUT_DIR)
from google.colab import files
files.download('/content/fine_tuned_v2.zip')
print('⬇️  Dézippe dans mental-health-signal-detector/models/fine_tuned/')
print('   puis relance : bash scripts/run_api.sh')